#Bag-Of-Words (BOW) e aprendizado não supervisionado
Nessa parte da disciplina, vamos analisar aplicações do modelo BOW para representação de conhecimento em documentos com técnicas de Aprendizado Não Supervisionado. Vamos analisar dois casos de uso que envolvem:
*   Agrupamento (Clusterização) de documentos;
*   Topicalização.

Ambos os processos envolvem a execução dos mesmos passos iniciais que foram realizados no processo de Classificação de Documentos com Aprendizado Supervisionado:
1.   Pré-processamento, para redução de dimensionalidade (com a lematização, por exemplo);
2.   Extração de Características, com as Classes Vetorizadoras (como TfidfVectorizer), filtragem de stop-words e, possivelmente, N-Gramas.

No entanto, em seguida, utilizaremos um algoritmo de Aprendizado Não-Supervisionado.

Esses processos de Agrupamento e Topicalização são frequentemente utilizados para identificar Documentos Similares e identificar Tópicos entre os documentos. Ambos esses processos podem ser utilizados na implementação de Sistemas de Recomendação de Conteúdo para usuários de uma aplicação.

Inicialmente, vamos fazer o download do mesmo dataset que temos utilizado em nossos estudos até agora:





In [1]:
import requests, os

url = "https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/buscape.csv"

def download (url):
  if (os.path.isfile('./buscape.csv')):
    print('Arquivo já existente no Runtime... Tudo OK')
    return
  response = requests.get(url)
  with open('./buscape.csv', 'wb') as f:
      f.write(response.content)
      print('Download realizado e arquivo extraído no Runtime... Tudo OK')

download(url)

Download realizado e arquivo extraído no Runtime... Tudo OK


#Pré-processamento e Extração de Características
Em seguida, vamos conduzir o pré-processamento com filtragem de linhas com dados faltantes, Lematização dos termos (#1) e a extração de características, removendo as stop-words, (#2) de forma similar aos passos conduzidos na aula passada.

Para simplificar, nesse exemplo, vamos analisar apenas os documentos que possuem polaridade negativa. Um outro ponto a ser destacado é que, como estamos trabalhando com uma tarefa de análise dos dados, não é necessário dividir o dataset em treino e teste (train_test_split).

```sh
pip install --upgrade spacy
python -m spacy download pt_core_news_sm
```

In [2]:


import spacy, pandas as pd
from spacy.lang.pt import stop_words
from sklearn.feature_extraction.text import TfidfVectorizer

pln = spacy.load("pt_core_news_sm", disable=["morphologizer", "senter", "attribute_ruler", "ner"])

df = pd.read_csv('./buscape.csv')                                        # (1)
df_filtered = df.loc[df['polarity']==0].dropna()
corpus_raw = df_filtered['review_text'].tolist()

def preprocessing (corpus):
  corpus_with_lemmas = []

  for text in corpus:
    doc = pln(text.lower())
    corpus_with_lemmas.append(' '.join([token.lemma_ for token in doc]))

  return corpus_with_lemmas

corpus_preprocessed = preprocessing(corpus_raw)

extractor = TfidfVectorizer(                                             # (2)
    stop_words=list(stop_words.STOP_WORDS))
X = extractor.fit_transform(corpus_preprocessed)

A partir da execução desses passos, Técnicas de Aprendizado Não Supervisionado podem ser utilizadas para as tarefas de Agrupamento e Topicalização.

#Agrupamento de Documentos com K-Means
O algoritmo K-Means é um método de clustering não supervisionado que agrupa dados em K clusters com base na similaridade. O K-Means começa com K pontos aleatórios (centroides) no espaço de dados. Em seguida, atribui cada ponto de dados ao centroide mais próximo, formando K clusters. Então, recalcula os centroides como os pontos médios de cada cluster. Esse processo é repetido até que os centroides não se movam mais ou um número máximo de iterações seja atingido.

A seguir, vamos treinar o nosso modelo baseado no algoritmo K-Means para gerar os clusters.

In [3]:
from sklearn.cluster import KMeans

model = KMeans(n_init='auto')
model.fit(X)


,"n_clusters n_clusters: int, default=8The number of clusters to form as well as the number ofcentroids to generate.For an example of how to choose an optimal value for `n_clusters` refer to:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_silhouette_analysis.py`.",8
,"init init: {'k-means++', 'random'}, callable or array-like of shape (n_clusters, n_features), default='k-means++'Method for initialization:* 'k-means++' : selects initial cluster centroids using sampling based on an empirical probability distribution of the points' contribution to the overall inertia. This technique speeds up convergence. The algorithm implemented is ""greedy k-means++"". It differs from the vanilla k-means++ by making several trials at each sampling step and choosing the best centroid among them.* 'random': choose `n_clusters` observations (rows) at random from data for the initial centroids.* If an array is passed, it should be of shape (n_clusters, n_features) and gives the initial centers.* If a callable is passed, it should take arguments X, n_clusters and a random state and return an initialization.For an example of how to use the different `init` strategies, see:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_digits.py`.For an evaluation of the impact of initialization, see the example:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_stability_low_dim_dense.py`.",'k-means++'
,"n_init n_init: 'auto' or int, default='auto'Number of times the k-means algorithm is run with different centroidseeds. The final results is the best output of `n_init` consecutive runsin terms of inertia. Several runs are recommended for sparsehigh-dimensional problems (see :ref:`kmeans_sparse_high_dim`).When `n_init='auto'`, the number of runs depends on the value of init:10 if using `init='random'` or `init` is a callable;1 if using `init='k-means++'` or `init` is an array-like... versionadded:: 1.2 Added 'auto' option for `n_init`... versionchanged:: 1.4 Default value for `n_init` changed to `'auto'`.",'auto'
,"max_iter max_iter: int, default=300Maximum number of iterations of the k-means algorithm for asingle run.",300
,"tol tol: float, default=1e-4Relative tolerance with regards to Frobenius norm of the differencein the cluster centers of two consecutive iterations to declareconvergence.",0.0001
,"verbose verbose: int, default=0Verbosity mode.",0
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation for centroid initialization. Usean int to make the randomness deterministic.See :term:`Glossary `.",None
,"copy_x copy_x: bool, default=TrueWhen pre-computing distances it is more numerically accurate to centerthe data first. If copy_x is True (default), then the original data isnot modified. If False, the original data is modified, and put backbefore the function returns, but small numerical differences may beintroduced by subtracting and then adding the data mean. Note that ifthe original data is not C-contiguous, a copy will be made even ifcopy_x is False. If the original data is sparse, but not in CSR format,a copy will be made even if copy_x is False.",True
,"algorithm algorithm: {""lloyd"", ""elkan""}, default=""lloyd""K-means algorithm to use. The classical EM-style algorithm is `""lloyd""`.The `""elkan""` variation can be more efficient on some datasets withwell-defined clusters, by using the triangle inequality. However it'smore memory intensive due to the allocation of an extra array of shape`(n_samples, n_clusters)`... versionchanged:: 0.18 Added Elkan algorithm.. versionchanged:: 1.1 Renamed ""full"" to ""lloyd"", and deprecated ""auto"" and ""full"". Changed ""auto"" to use ""lloyd"" instead of ""elkan"".",'lloyd'


Para facilitar nossas análises, vamos utilizar um DataFrame para armazenar o texto dos documentos (puro e pré-processado) e os clusters aos quais esses documentos pertencem (disponibilizado no atributo model.labels_).

In [5]:
clusters_df = pd.DataFrame()
clusters_df['text'] = corpus_raw
clusters_df['text_preprocessed'] = corpus_preprocessed
clusters_df['cluster'] = model.labels_

clusters_df

,text,text_preprocessed,cluster
0,"nao comparia novamente essa marca, pois paguei...","nao compar novamente esse marca , pois paguei ...",7
1,tem uma boa aderencia te da mas agilidades no ...,ter um bom aderencia te de o mas agilidade em ...,3
2,"Somente mais um jogo do estilo, nada realmente...","somente mais um jogo de o estilo , nada realme...",3
3,Estou insatisfeito com o produto porque a Elet...,estar insatisfeito com o produto porque ela el...,6
4,"SANSUNG É MINHA PIOR COMPRA que já fiz, não re...","sansung ser meu mau compra que já fiz , não re...",7
...,...,...,...
6805,Ele e um celular funcional para quem deseja mu...,ele e um celular funcional para quem desejar m...,0
6806,Produtos NKS. Nunca mais. Uma porcaria e o SAC...,produto nks . nunca mais . um porcaria e o sac...,6
6807,Comprei este por ele ser mais acessivel (valor...,comprei este por ele ser mais acessivel ( valo...,3
6808,comprei o ar condicionado a pouco mais de um a...,comprei o ar condicionado a pouco mais de um a...,3


Com os clusters definidos, nós podemos, dado um documento, procurar por documentos similares dentro do mesmo cluster.

In [6]:
doc = clusters_df.iloc[77]
print(doc['text'])
print(doc['cluster'])

neutro!

O que não gostei: qualidade de som ruin!
3


Nesse exemplo, nós selecionamos o documento da linha 77 do corpus. Esse documento menciona uma perspectiva negativa com relação a qualidade de som de algum produto.

A seguir, a partir do cluster desse documento, podemos procurar por documentos que pertencem ao mesmo cluster. Nesse exemplo específico, nós podemos observar que existem outras postagens no mesmo cluster que também mencionam problemas na qualidade de som. Como exercício, vocês podem experimentar outros documentos para verificar se os clusters armazenam informações relevantes e relacionadas ao tema definido (por exemplo, com o documento do índice 130).

Podem existir clusters que não possuem um relacionamento claro entre os documentos. Isso pode estar relacionado a limitações da abordagem BOW em mapear as informações ou as configurações de número de clusters utilizada no algoritmo K-Means. Nesse sentido, vocês podem explorar executar o K-Means com outros parâmetros para verificar os diferentes agrupamentos que podem ser estabelecidos.

In [7]:
doc_cluster = clusters_df.loc[clusters_df['cluster'] == doc['cluster']]
print('\n \n --- \n \n'.join(doc_cluster['text'].head().tolist()))

tem uma boa aderencia te da mas agilidades no dribles e confortavel

O que gostei: tem uma boa aderencia te da mas agilidades no dribles e confortavel

O que não gostei: tem uma boa aderencia te da mas agilidades no dribles e confortavel
 
 --- 
 
Somente mais um jogo do estilo, nada realmente marcante.

O que gostei: Diversão.

O que não gostei: Falta de originalidade, de maior liberdade, de história mais elaborada e melhor jogabilidade.
 
 --- 
 
Olá a todos. Comprei o coffret que está na promoção na Sack´s. Chegou dentro do prazo e as amostras grátis vieram certinhas como escolhi. Ou seja, nota dez pra Sack´s. Agora quanto ao Guerlain Homme Masculino, achei uma fragrância muito fraca. Tenho que colocar muito para senti-lo presente em mim e com isso não dura nada. Acho indicada para pessoas que gostam de fragrâncias discretas, suaves do tipo que só quando se chega perto é que se sente, pois não é de deixar rastro. É uma fragrância boa cítrica amadeirada e, diga-se de passagem, é muit

O modelo gerado também pode ser utilizado para identificar o cluster de um novo documento. Nesse exemplo, vamos considerar que temos uma postagem com o texto identificado no código a seguir. Vamos realizar as seguintes operações com esse texto, para identificar o seu cluster:
1.   Executar o pré-processamento para substituir as palavras por seus lemas;
2.   Extrair as características, removendo as stop-words;
3.   Identificar o cluster ao qual o documento pertence;
4.   Selecionar outros documentos do mesmo cluster.



In [8]:
doc_novo = "Não comprem esse celular. Não funciona bem, a bateria não dura nada e a tela é monocromática."

doc_novo_preprocessed = preprocessing([doc_novo])                       # (1)
doc_novo_features = extractor.transform(doc_novo_preprocessed)          # (2)
[cluster_id] = model.predict(doc_novo_features)                         # (3)
doc_cluster = clusters_df.loc[clusters_df['cluster'] == cluster_id]     # (4)
print('\n \n --- \n \n'.join(doc_cluster['text'].head().tolist()))

pessimo ruim mais bonito lento  poucos aplicativos

O que gostei: bontoe com camara

O que não gostei: ruim  lento fragio e com poucos aplicativos
 
 --- 
 
Qualidade do som deprimente. Dificil ter conversas com esse telefone. Péssimo para escutar e para falar. Menu não-amigável. Despertador complicado. Péssimo para salvar numeros, chamadas e enviar mensagens. Quem já teve um nokia, jamais vai querer um celular desse.

O que gostei: Barato.

O que não gostei: Qualidade do som deprimente. Péssimo para escutar e para falar. Menu não-amigável.
 
 --- 
 
NÃO ACONSELHO A COMPRA PORQUE O QUE ADIANTA TER CÂMERA CARTÃO DE MEMÓRIA 2 GB,  TER RÁDIO, MÚSICA ETC, E  "NÃO PODER USAR PORQUE A BATERIA NÃO DURA NADA UM VERDADEIRO LIXO, A MOTOROLA DEVERIA TER VERGONHA NA CARA AO OFERECER UM PORCARIA DESSA.
NÃO COMPREM MOTOROLA,  PALAVRA DE QUEM SE FERROU.

O que gostei: MUITO BONITO. CÂMERA 3.1

O que não gostei: BATERIA RIDICULAMENTE FRACA, NÃO DURA NADA, UM LIXO. O SOM É RUIM TANTO PARA FALAR COMO PA

#Latent Dirichlet Allocation (LDA)
O algoritmo Latent Dirichlet Allocation (LDA) é um modelo estatístico generativo usado para descobrir tópicos latentes em um corpus de texto. Cada documento é representado como uma mistura de tópicos, onde os tópicos são distribuições de probabilidade sobre as palavras. O LDA é uma técnica popular de modelagem de tópicos que ajuda a identificar os temas subjacentes em um conjunto de documentos.

Para determinar os valores necessários para geração de tópicos, o LDA precisa apenas da contagem das palavras. Por isso, podemos utilizar o CountVectorizer para extrair as características.

In [9]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation as LDA

extractor_count = CountVectorizer(
    binary=True, stop_words=list(stop_words.STOP_WORDS))
X_lda = extractor_count.fit_transform(corpus_preprocessed)

model = LDA()
model.fit(X_lda)

print(model.components_.shape)
print(model.components_)

(10, 16402)
[[ 0.70033615  8.10151757  0.100004   ...  0.1         0.10002531
   0.1       ]
 [10.34622094 14.66085196  0.10002702 ...  0.10002337 15.58054484
   0.1       ]
 [ 4.61163365  0.10001808  0.1        ...  0.1         0.10000833
   0.1       ]
 ...
 [ 6.18529431  0.10014567  0.10000145 ...  0.1        23.28247348
   1.09992694]
 [ 6.37689091  0.10000523  0.10000471 ...  0.1         0.10001006
   0.1       ]
 [52.22572962  5.96764371  1.15800403 ...  0.1        12.42706486
   0.1       ]]


O atributo model.components_ contém a representação dos tópicos gerados pelo LDA. Por padrão, o LDA gera 10 tópicos para o texto (isso pode ser configurável no construtor da classe). Cada tópico (cada linha do atributo model.components_) possui um valor de "peso" associado a cada palavra do córpus.

Para entender os tópicos gerados pelo algoritmo, nós podemos identificar as palavras que possuíram maior impacto na definição dos tópicos, implementando os seguintes passos:
1.   Para cada tópico, identificar os 10 termos que possuem maior representação nos tópicos (isso deve identificar índices dos termos mais importantes por tópico);
2.   Recuperar o termo (palavra) associada ao índice do termo do dicionário de termos do extrator de características;
3.   Para facilitar, análises futuras, vamos armazenar esses termos, que identificam os tópicos, em uma vetor.

In [14]:
terms_dict = extractor_count.get_feature_names_out()
topic_top_terms = []

for id, topic in enumerate(model.components_):
  ind_sorted = topic.argsort()                         # (1)
  terms = [ terms_dict[ind] for ind in ind_sorted ]    # (2)

  print(f'Topic {id}: {terms[-10:]}')

  topic_top_terms.append(terms[-10:])

Topic 0: ['nao', 'outro', 'pequeno', 'bonito', 'comprar', 'ficar', 'qualidade', 'preço', 'produto', 'gostar']
Topic 1: ['informar', 'empresa', 'funcionar', 'assistência', 'gostar', 'produto', 'técnico', 'problema', 'comprei', 'dia']
Topic 2: ['comprei', 'marca', 'outro', 'som', 'produto', 'nao', 'comprar', 'imagem', 'tv', 'gostar']
Topic 3: ['peça', 'ano', 'dia', 'problema', 'uso', 'funcionar', 'produto', 'assistência', 'técnico', 'gostar']
Topic 4: ['outro', 'marca', 'comprei', 'preço', 'comprar', 'qualidade', 'imagem', 'tv', 'produto', 'gostar']
Topic 5: ['00', 'cliente', 'nao', 'produto', 'geladeira', 'gostar', 'passar', 'porta', 'problema', 'defeito']
Topic 6: ['problema', 'loja', 'nao', 'ficar', 'qualidade', 'preço', 'outro', 'comprar', 'produto', 'gostar']
Topic 7: ['trar', 'durar', 'ruim', 'produto', 'tela', 'outro', 'celular', 'aparelho', 'bateria', 'gostar']
Topic 8: ['conseguir', 'modelo', 'consumidor', 'outro', 'marca', 'criança', 'vir', 'hora', 'produto', 'gostar']
Topic 9:

Nesse exemplo, nós podemos ver os termos que mais influenciaram a geração dos tópicos. Podemos observar tópicos que estão associados à bateria de celular, TV com defeito no som, celulares com problema de som, perfume, entre outros.

Dado um documento específico, também podemos identificar os tópicos associados à esse documento.

In [ ]:
docs_test = corpus_raw[42:47]            # pegando um conjunto de documentos
docs_test_preprocessed = preprocessing(docs_test)
docs_test_features = extractor_count.transform(docs_test_preprocessed)
docs_test_topics = model.transform(docs_test_features)

print(docs_test_topics)

[[0.00500166 0.00500255 0.00500075 0.13539802 0.78242193 0.00500048
  0.00500049 0.0471717  0.00500128 0.00500114]
 [0.01111496 0.0111118  0.01111451 0.01111241 0.01111393 0.01111198
  0.0111129  0.60580801 0.01111194 0.30528756]
 [0.008337   0.13189457 0.00833457 0.00833583 0.0083345  0.00833482
  0.00833406 0.00833475 0.12603513 0.68372477]
 [0.88748378 0.01250017 0.01250181 0.01250111 0.01250362 0.01250344
  0.01250057 0.01250209 0.01250113 0.01250228]
 [0.28226482 0.00526365 0.00526377 0.00526398 0.00526503 0.00526337
  0.00526435 0.67562285 0.0052642  0.00526398]]


Para cada documento, foi gerado um vetor com 10 elementos. Cada elemento representa um tópico gerado pelo LDA. Nesse vetor, os valores representam probabilidades de que um tópico específico esteja associado ao documento.

Para facilitar a visualização desses resultados, vamos recuperar os 3 tópicos que possuem maior probabilidade de relação com cada documento, se a sua probabilidade for maior que 0.2. Em seguida, vamos imprimir o texto do documento, juntamente com esses tópicos.

In [16]:
for i, doc_topic in enumerate(docs_test_topics):
  ind_topic_sorted = doc_topic.argsort()
  ind_topic_top = [ i for i in ind_topic_sorted[-3:] if doc_topic[i] > 0.1 ]

  print(f'------\n{docs_test[i]}\n\n  Tópicos: {ind_topic_top}')
  for ind_topic in ind_topic_top:
    print(f'   - Tópico {ind_topic}: {topic_top_terms[ind_topic]}')

  print('\n\n')

------
Comprei o carregador, a entrega fui muito rápida.

Quando peguei o carregador, li e segui as instruções, na primeira vez que coloquei o carregador na tomada, ele deu uma pequena explosão e não funciona mais.

Agora estou tentando entrar em contato com a empresa.

O que gostei: Não vi ainda

O que não gostei: Queimou, não funciona no primeiro uso.

  Tópicos: [np.int64(3), np.int64(4)]
   - Tópico 3: ['peça', 'ano', 'dia', 'problema', 'uso', 'funcionar', 'produto', 'assistência', 'técnico', 'gostar']
   - Tópico 4: ['outro', 'marca', 'comprei', 'preço', 'comprar', 'qualidade', 'imagem', 'tv', 'produto', 'gostar']



------
boa tarde, esse aparelho tem wi - fi? grata

O que gostei: bonito

O que não gostei: nao conheço o aparelho

  Tópicos: [np.int64(9), np.int64(7)]
   - Tópico 9: ['defeito', 'marca', 'outro', 'assistência', 'técnico', 'funcionar', 'ficar', 'produto', 'problema', 'gostar']
   - Tópico 7: ['trar', 'durar', 'ruim', 'produto', 'tela', 'outro', 'celular', 'aparelho'

Nesse exemplo, nós podemos observar que nem todos os termos e tópicos fazem sentido com o texto do documento. No entanto, não estamos analisando todos os termos que compõem o documento, apenas os 10 primeiros. O dicionário de termos pode ter muitos outros termos que influenciam na definição dos tópicos.

Os tópicos 8 e 6 estão relacionados ao uso de termos positivos no documento e foram associados aos documentos que possuem essa mesma conotação.

Os tópicos 3 e 2 aprensentam termos que denotam problemas na assistência técnica, enquanto o tópico 5 está relacionado a fragrância e perfumes.

#Considerações finais
O uso de Aprendizado Não Supervisionado é frequente realizado para analisar dados textuais. Nos exemplos de aplicações, as análises foram realizadas para agrupar documentos similares e identificar tópicos entre os documentos.

Essas análises são frequentemente utilizadas para gerar insights em uma elevada quantidade de documentos textuais. Identificar essas informações de forma manual, com um corpus composto por 70 mil documentos, seria uma tarefa difícil de ser realizada.

Foi utilizado o modelo Bag-Of-Words para representar o conhecimento dos documentos, mas poderiam ser utilizados outros modelos. As análises poderiam ser refinadas, alterando configurações das etapas de pré-processamento, extração de características, seleção de características, transformações nos dados (redução de dimensionalidade) e otimização dos hiper parâmetros dos modelos de Aprendizado Não Supervisionado.